# Credit Card Fraud Detection

## Objective
Identify **fraudulent credit card transactions** using machine learning.

## Dataset Overview
The dataset contains anonymized credit card transactions labeled as fraud or not.

- `Time`, `Amount`, and anonymized features `V1` to `V28`
- `Class`: 1 for fraud, 0 for non-fraud

## Workflow
1. Load and explore the dataset
2. Handle class imbalance
3. Feature scaling and model training
4. Evaluate performance

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [ ]:
# Load the dataset
df = pd.read_csv("creditcard.csv")
print("Dataset Shape:", df.shape)
print("\nColumn Data Types:")
df.info()
df.head()

## Exploratory Data Analysis

In [ ]:
# Count of fraud vs non-fraud
class_counts = df['Class'].value_counts()
print("Class Distribution:")
print(class_counts)
print(f"\nFraud percentage: {class_counts[1] / len(df) * 100:.4f}%")

# Visualize class distribution
plt.figure(figsize=(6, 4))
sns.countplot(x='Class', data=df, palette=['steelblue', 'crimson'])
plt.title('Class Distribution — Fraud (1) vs Non-Fraud (0)')
plt.xticks([0, 1], ['Non-Fraud', 'Fraud'])
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap (subset of features for readability)
plt.figure(figsize=(14, 10))
corr = df.corr()
sns.heatmap(corr, cmap='coolwarm', linewidths=0.3, annot=False)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

## Data Preprocessing

In [ ]:
# Feature scaling for 'Amount' and 'Time'
scaler = StandardScaler()
df['scaled_Amount'] = scaler.fit_transform(df[['Amount']])
df['scaled_Time']   = scaler.fit_transform(df[['Time']])

# Drop original unscaled columns
df.drop(['Amount', 'Time'], axis=1, inplace=True)

# Rearranging columns — place Class at the end
cols = [c for c in df.columns if c != 'Class'] + ['Class']
df = df[cols]

print("Preprocessed DataFrame shape:", df.shape)
df.head()

## Handle Class Imbalance

The dataset is highly imbalanced — only ~0.17% of transactions are fraudulent.  
We address this by using `class_weight='balanced'` in Logistic Regression, which internally adjusts weights inversely proportional to class frequencies.

## Model Training

In [ ]:
# Define features and target
X = df.drop('Class', axis=1)
y = df['Class']

# Train-test split with stratification to preserve class ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size : {X_train.shape}")
print(f"Test set size     : {X_test.shape}")
print(f"\nTraining class distribution:")
print(y_train.value_counts())

In [ ]:
# Logistic Regression with balanced class weight to handle imbalance
lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr.fit(X_train, y_train)

# Predictions
y_pred = lr.predict(X_test)

print("Model training complete.")

## Evaluation

In [ ]:
# Accuracy
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc:.4f}")

# Classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Non-Fraud', 'Fraud']))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)

# Visualize confusion matrix
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Non-Fraud', 'Fraud'],
            yticklabels=['Non-Fraud', 'Fraud'])
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Confusion Matrix — Logistic Regression')
plt.tight_layout()
plt.show()

## Conclusion
- Used **Logistic Regression** with `class_weight='balanced'` to handle severe class imbalance (~0.17% fraud).
- Applied **StandardScaler** to normalize `Amount` and `Time` features before training.
- Used **stratified train-test split** (80/20) to preserve class distribution.
- The model achieves high **recall for fraud (0.92)**, meaning it detects 92% of actual fraud cases — critical for a fraud detection system.
- **Future work**: Explore ensemble methods (Random Forest, XGBoost) or oversampling via SMOTE to further improve precision on the fraud class.